# **Calidad del Aire en Colombia: Análisis y Predicción** 
**Por:** Juan Esteban Hernandez Gualtero

## **Fase 1: Tema, Justificación y Preguntas Clave**

### **El Tema: Análisis Territorial y Temporal de la Calidad del Aire en Colombia.**
El dataset contiene registros de variables críticas (como material particulado $PM_{10}$, $PM_{2.5}$, dirección del viento, etc.) monitoreadas por distintas autoridades ambientales en múltiples municipios del país.

### **Justificación:**
La contaminación atmosférica es uno de los principales riesgos ambientales para la salud pública en Colombia. Analizar este dataset no solo permite mapear los focos críticos de contaminación, sino también evaluar la confiabilidad de las mediciones (representatividad temporal) y el comportamiento de los contaminantes frente a los límites legales actuales. Esto es fundamental antes de entrenar cualquier modelo predictivo, ya que el comportamiento geográfico y la naturaleza de cada contaminante dictarán la estrategia de modelado.

### **Preguntas Clave:**
- Distribución de Contaminantes: ¿Cuáles son las variables más medidas en el país y qué regiones (Departamentos/Municipios) presentan los promedios anuales más altos de contaminantes críticos como $PM_{2.5}$ y $PM_{10}$?
- Cumplimiento Normativo: ¿Qué porcentaje de las estaciones reportan excedencias de los límites legales (Excedencias limite actual) y en qué tipo de estaciones (Fijas vs. Móviles) es más frecuente este fenómeno?
- Comportamiento de Extremos: ¿Qué relación existe entre la mediana, el percentil 98 y los valores máximos registrados para los principales contaminantes?

## **Fase 2: Adquisición y Comprensión de Datos**

### **Fuente de los Datos:**
https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Calidad-del-Aire-en-Colombia-Promedio-Anual-/kekd-7v7h

### **Carga y primera inspección del dataset:**
En este punto, se carga el dataset para planear un panorama general, y tener una primera visualización de la información con la que se estará trabajando a lo largo de las fases.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Calidad_del_Aire_en_Colombia_(Promedio_Anual)_20260622.csv')

# 1. Inspeccion de la forma
print("--- Dimensiones del Dataset ---")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print("\n")

# 2. Visualizacion rapida
print("--- Primeras 5 filas ---")
display(df.head())

# 3. Info general: tipos de datos y valores no nulos
print("\n--- Información General y Tipos de Datos ---")
print(df.info())

### **Primeros vistazos:**
Se dispone de un dataset con numerosos registros (mas de 10k). Hay en total 9 columnas numéricas (int y float), y 19 de texto (string). A simple vista parece ser un dataset con pocos valores nulos entre sus registros a pesar de su tamaño, lo que podria facilitar el desarrollo de las fases posteriores.


## **Fase 3: Diagnóstico de Calidad de Datos**

In [ ]:
print("==========================================================")
print("                    EJECUTANDO DIAGNOSTICO                ")
print("==========================================================\n")

# 1. Identificar columnas numericas catalogadas como string
columnas_numericas_atrapadas = [
    'Promedio', 'Suma', 'No. de datos', 'Mediana', 
    'Percentil 98', 'Máximo', 'Mínimo', 'ID Estacion'
]

print("1. ANALISIS DE CARACTERES ESPECIALES EN COLUMNAS NUMERICAS:")
print("-" * 60)
for col in columnas_numericas_atrapadas:
    if col in df.columns:
        # Verificar si la columna se cargo como objeto/string
        es_objeto = df[col].dtype == 'object'
        
        # Contar cuantos registros contienen comas (separadores de miles propensos a romper Pandas)
        con_coma = df[col].astype(str).str.contains(',').sum()
        
        # Contar cuantos contienen texto o letras (ej: "ND", "Sin dato")
        con_texto = df[col].astype(str).str.contains('[a-zA-Z]', regex=True).sum()
        
        print(f"Columna: '{col}' (Tipo original: {df[col].dtype})")
        print(f"  -> Registros con comas: {con_coma} ({con_coma/len(df)*100:.2f}%)")
        print(f"  -> Registros con texto/letras: {con_texto} ({con_texto/len(df)*100:.2f}%)")
        print("-" * 60)

# 2. Busqueda de nulos camuflados (espacios en blanco)
print("\n2. DETECCION DE ESPACIOS EN BLANCO:")
print("-" * 60)
for col in df.columns:
    if df[col].dtype == 'object':
        espacios_blanco = df[col].astype(str).str.strip().eq('').sum()
        if espacios_blanco > 0:
            print(f"  - Columna '{col}': {espacios_blanco} registros son solo espacios vacios.")


# 3. Revision de consistencia en registros geograficos e ID
print("\n3. CONSISTENCIA GEOGRAFICA Y CATEGORICA:")
print("-" * 60)
nulos_municipio_id = df['Código del Municipio'].isnull().sum()
nulos_municipio_nom = df['Nombre del Municipio'].isnull().sum()
print(f"  - Registros sin Código de Municipio: {nulos_municipio_id}")
print(f"  - Registros sin Nombre de Municipio: {nulos_municipio_nom}")
print(f"  - Años unicos registrados en el dataset: {df['Año'].unique()}")
print(f"  - Cantidad de contaminantes/variables distintas: {df['Variable'].nunique()}")

Tras inspeccionar la estructura general del dataset, se han identificado anomalías críticas que impiden el análisis estadístico directo y el entrenamiento de modelos de Machine Learning. A continuación, se detallan los hallazgos:

### **1. Variables Cuantitativas en Formato de texto**
Las variables esenciales para el estudio de la calidad del aire (Promedio, Suma, Mediana, Percentil 98, Máximo y Mínimo) fueron interpretadas por Pandas como tipo object (texto) en lugar de float64, aunque no se encontraron registros con datos alfanuméricos dentro de las columnas analizadas.

- Causa principal: El uso de comas (,) como separadores de miles en el archivo de origen (por ejemplo, valores como "321,637.50" o "4,418.76"). Al no reconocer la coma como un formato puramente numérico estándar, Python procesa toda la columna como una cadena de caracteres.

- Impacto: Imposibilidad de calcular distribuciones, correlaciones o matrices de covarianza de forma nativa.

### **2. Nulos Camuflados**
Además de las comas, existen registros en las columnas numéricas que contienen caracteres alfabéticos directos (letras). Esto suele corresponder a convenciones institucionales para reportar datos faltantes (como "ND", "S.D." o "No Registra"). Igualmente, espacios en blanco suelen afectar el análisis, al parecer que si se ingresaron los datos, cuando solo hay un espacio vacío (" "). Los anteriores casos actúan como "nulos ocultos" que distorsionan el conteo inicial de df.isnull().sum(). Por fortuna, no se encontraron registros que dieran con la presencia de algunos de estos problemas.

### **3. Inconcistencia Geográfica**
Existe una discrepancia menor en la completitud de la ubicación: la columna Código del Municipio presenta 4 valores nulos absolutos, mientras que Nombre del Municipio está 100% diligenciado. Esto evidencia un problema de consistencia en el cruce de diccionarios de datos geográficos de origen.



## **4. Limpieza de Datos**

In [ ]:
# Crear una copia para trabajar la limpieza
df_clean = df.copy()

print("==========================================================")
print("                      LIMPIEZA DE DATOS                    ")
print("==========================================================\n")

# 1. Definir columnas que deben ser puramente numericas
columnas_a_convertir = [
    'Promedio', 'Suma', 'No. de datos', 'Mediana', 
    'Percentil 98', 'Máximo', 'Mínimo', 'Año', 'ID Estacion'
]

# 2. Proceso de limpieza y casteo
for col in columnas_a_convertir:
    if col in df_clean.columns:
        # Convertir a string, limpiar espacios en blanco y eliminar comas
        df_clean[col] = df_clean[col].astype(str).str.strip().str.replace(',', '', regex=False)
        
        # Forzar a numerico. Espacios vacios o textos se vuelven NaN automaticamente
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 3. Ajustar tipos de datos para enteros que aceptan nulos (evita que se vuelvan floats con .0)
enteros_con_nan = ['Año', 'ID Estacion', 'No. de datos', 'Código del Municipio', 'Código del Departamento']
for col in enteros_con_nan:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('Int64')

# 4. VERIFICACION FINAL DE LIMPIEZA:
print("--- VERIFICACION DE NUEVOS TIPOS DE DATOS ---")
print(df_clean[columnas_a_convertir + ['Código del Municipio']].dtypes)
print("\n" + "-"*50)

print("\n--- CONTEO DE NULOS REALES (POST-LIMPIEZA) ---")
print("Ahora si se evidencia la verdadera cantidad de datos faltantes:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Para corregir las anomalías detectadas en la fase de diagnóstico, se aplicó un pipeline de transformación iterativo sobre las variables cuantitativas y geográficas:

- **Tratamiento de caracteres y espacios:**
 Se removieron los espacios en blanco espurios mediante desbroce de cadenas (strip) y se eliminaron los separadores de miles(,).

- **Conversión de tipos e imputación pasiva de nulos:**
 Se aplicó una conversión forzada a tipo numérico. Aquellos registros que contenían nulos camuflados (celdas vacías "" o marcadores de texto institucional) fueron coaccionados rigurosamente a valores NaN (Not a Number) nativos de Python.

- **Preservación de Estructura:** Las variables de identificación (como códigos geográficos, IDs y años) se formatearon bajo el tipo Int64 de Pandas para mitigar la conversión automática a flotantes debido a la presencia de valores faltantes.

**Estado del Dataset:** Las columnas ahora tienen propiedades matemáticas reales. Esto nos desvela el verdadero volumen de datos faltantes, permitiéndonos realizar un análisis exploratorio (EDA) estadísticamente riguroso sin sesgos por strings ocultos.

Para proseguir con la siguiente fase, es práctico eliminar estos registros con estas columnas en nulo, pues representan menos del 0.5% de los registros totales que hay (La Representatividad Temporal, justamente, solo un 0.49%) y no intervienen o generan un aporte significativo al resto del análisis y posterior modelo a entrenar.

In [ ]:
df_clean = df_clean.dropna()

print("--- Dimensiones Actualizadas del Dataset ---")
print(f"Filas: {df_clean.shape[0]}, Columnas: {df_clean.shape[1]}")
print("\n")

## **5. Análisis Exploratorio de Datos**

### **5.1 Análisis Univariado y Detección de Outliers**
Aquí se evalúa la distribución individual de los contaminantes principales. Con Diagramas de Caja (Boxplots), que son la herramienta perfecta para visualizar los cuartiles y exponer inmediatamente los outliers (todo punto que caiga fuera de los "bigotes" del gráfico). También histogramas para ver la forma de la curva (si está sesgada hacia valores de alta contaminación).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion estetica
sns.set_theme(style="whitegrid")

# Filtrar los contaminantes de mayor interes para no saturar los graficos
df_pm = df_clean[df_clean['Variable'].isin(['PM10', 'PM2.5'])]

# Crear figura para el analisis univariado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Boxplot para Deteccion de Outliers
sns.boxplot(x='Variable', y='Promedio', data=df_pm, palette="Set2", ax=axes[0])
axes[0].set_title("Detección de Outliers: PM10 y PM2.5 (Promedio)")
axes[0].set_ylabel("Concentración Promedio")

# 2. Histograma para Distribucion
sns.histplot(data=df_pm, x='Promedio', hue='Variable', bins=50, kde=True, ax=axes[1])
axes[1].set_title("Distribución de Concentraciones")
axes[1].set_xlabel("Concentración Promedio")

plt.tight_layout()
plt.show()

### **Hallazgos Clave:**
- Agrupación Masiva de Outliers (Boxplot):Efectivamente, se ve una línea densa y continua de diamantes negros por encima del bigote superior tanto para PM10 como para PM2.5. Para el PM10, la caja intercuartílica está muy comprimida abajo (probablemente entre 10 y 40 $\mu g/m^3$), pero hay registros que se disparan casi hasta 500 $\mu g/m^3$. Esto no es un error de digitación; es muy probable que estemos viendo el reflejo de contingencias ambientales, temporadas secas prolongadas, focos industriales pesados o incendios forestales.

- Asimetría Positiva Fuerte (Histograma):El gráfico de la derecha confirma lo del boxplot con una distribución fuertemente sesgada a la derecha (right-skewed). La inmensa mayoría de los municipios en Colombia respira un aire dentro de rangos normales/bajos (el pico alto a la izquierda), pero esa "cola larga" hacia la derecha nos muestra que hay zonas específicas con una contaminación crónica o picos muy agresivos.

- Si se aplican modelos lineales (como Regresión Lineal o Logística), esta cola derecha destrozará las métricas de error. Habría que aplicar una transformación logarítmica a la variable objetivo para normalizar la curva. Sin embargo, si la estrategia apunta a ensambles basados en árboles (como Random Forest o Gradient Boosting), estos algoritmos son excepcionalmente robustos frente a esta clase de distribuciones, ya que aislarán estos valores extremos en nodos específicos sin sesgar el resto de las predicciones.

### **5.2 Análisis Bivariado**
Se cruzarán variables para encontrar patrones. Por ejemplo: ¿Cómo varía la concentración promedio dependiendo del Departamento? Esto ayuda a responder las preguntas geográficas planteadas en la Fase 1.

In [ ]:
# Crear figura para el analisis
plt.figure(figsize=(16, 7))

# Grafico de barras por departamento
sns.barplot(
    x='Nombre del Departamento', 
    y='Promedio', 
    hue='Variable', 
    data=df_pm, 
    errorbar=None,
    palette="viridis"
)

plt.xticks(rotation=90)
plt.title("Promedio Anual de PM10 y PM2.5 por Departamento")
plt.ylabel("Concentración Promedio")
plt.xlabel("Departamento")
plt.tight_layout()
plt.show()

### **Hallazgos Clave:**
El gráfico de barras revela dos realidades ambientales completamente distintas en el país, divididas por el tipo de contaminante:

**El problema del PM10 (Material particulado grueso):**

Se observan picos agresivos en La Guajira y Cesar, superando con creces al resto de los departamentos. Esto tiene todo el sentido desde el punto de vista del negocio/dominio. Estos departamentos concentran la minería de carbón a cielo abierto más grande del país y zonas áridas, lo que levanta enormes cantidades de polvo y material grueso (PM10).

**El problema del PM2.5 (Material particulado fino):**

Los promedios más altos se concentran en Antioquia, Bogotá D.C. y Valle del Cauca. El PM2.5 está estrictamente ligado a la combustión (gases de escape de vehículos e industria). Antioquia (específicamente el Valle de Aburrá) y Bogotá sufren por la alta densidad vehicular y su topografía de valle/sabana, que "atrapa" este contaminante fino, elevando drásticamente el promedio anual.

### **5.3 Análisis Multivariado**
Lo principal es una Matriz de Correlación. Esto es vital antes del Machine Learning, ya que si dos variables están perfectamente correlacionadas (ej. Máximo y Percentil 98), se podría eliminar una para evitar la multicolinealidad en el modelo.

In [ ]:
# Seleccionar unicamente columnas numericas de interes
cols_corr = [
    'Promedio', 
    'Máximo', 
    'Mínimo', 
    'Representatividad Temporal', 
    'Excedencias limite actual'
]

# Calcular matriz de correlacion de Pearson
corr_matrix = df_clean[cols_corr].corr()

# Generar el Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm', 
    vmin=-1, 
    vmax=1, 
    fmt=".2f",
    linewidths=0.5
)

plt.title("Matriz de Correlación - Variables Numéricas")
plt.tight_layout()
plt.show()

### **Hallazgos Clave:**
El mapa de calor nos da advertencias críticas (alertas rojas) y tranquilidad (alertas verdes) antes de entrenar modelos:

**Alerta Roja (Multicolinealidad y Fuga de Datos):**

Existe una fuerte correlación positiva entre Máximo, Excedencias limite actual y Promedio (valores entre 0.69 y 0.71). Si se quiere entrenar un modelo de regresión para predecir el Promedio de contaminación, no se puede incluir el Máximo o las Excedencias como variables predictoras (features). Hacerlo generaría Data Leakage (fuga de información), ya que estas variables son esencialmente consecuencias matemáticas unas de otras.

**Alerta Verde (Integridad de la Medición):**

La Representatividad Temporal tiene una correlación prácticamente nula con todas las variables de contaminación (ej. 0.051 con el Promedio). Esto significa que los sensores miden de forma constante independientemente de si el aire está limpio o sucio. Descarta el sesgo de que las estaciones "fallen" o se apaguen justo cuando hay picos de contaminación.

## **Fase 6: Finalización del EDA**

Para cerrar formalmente el Análisis Exploratorio, se consolidan las conclusiones respondiendo a las preguntas iniciales de la Fase 1:

- Distribución de Contaminantes: La contaminación no es homogénea. El riesgo por PM10 es territorial (zonas mineras/áridas en la región Caribe), mientras que el riesgo por PM2.5 es puramente demográfico/industrial (grandes urbes en la región Andina y Pacífica).

- Calidad de la Medición: La constancia en la recolección de datos (Representatividad Temporal) es independiente de los niveles de contaminación, lo que da confiabilidad estadística al dataset limpio.

- Comportamiento de Extremos: Los datos presentan una fuerte asimetría positiva con múltiples outliers válidos. Las zonas con altos promedios anuales están directamente correlacionadas con la ocurrencia de picos extremos diarios y violaciones a los límites legales.

## **Fase 7: Planeación del Entrenamiento de Modelos**

Para las fases 8, 9 y 10, se requiere elegir tres algoritmos distintos que nos permitan atacar el problema desde diferentes perspectivas. Dado que la variable óptima predecir (Promedio) es un número continuo, estamos frente a un problema de Regresión. Además, por el EDA se sabe que hay muchos outliers válidos y asimetría.

Aquí los tres candidatos que se entrenarán y el porqué metodológico de cada uno:

### **Regresión Lineal Múltiple (El modelo base):**
- Es el algoritmo más clásico. Intenta trazar una línea recta (o un hiperplano) a través de los datos que minimice la distancia entre la línea y los puntos reales.

- ¿Por qué? En Machine Learning siempre se debe empezar por lo simple. Será el "modelo base" (baseline). Si un modelo complejo no puede superar a una simple línea recta, no vale la pena usarlo. Además, es altamente interpretable (se sabrá exactamente qué peso le da a cada variable).

- El reto: Sufre mucho con los outliers y asume que la relación entre variables es recta, lo cual rara vez ocurre en la naturaleza.

### **Random Forest Regressor (El poder del ensamble):**
- En lugar de una línea, este modelo crea cientos de "Árboles de Decisión" independientes. Cada árbol hace su propia predicción dividiendo los datos con preguntas de "sí/no" (ej. ¿Es de La Guajira? ¿Es PM10?). Al final, el modelo promedia las respuestas de todos los árboles.

- Es brutalmente efectivo contra los outliers (los aísla en ramas separadas) y no le importa si los datos no siguen una distribución normal o lineal. Es ideal para la forma que vimos en nuestros histogramas.

### **Gradient Boosting Regressor (El perfeccionista):**
- También usa árboles de decisión, pero con una filosofía distinta. En lugar de crear cientos de árboles independientes al mismo tiempo, los crea en secuencia. El primer árbol hace una predicción; el segundo árbol analiza los errores (residuales) del primero y trata de corregirlos; el tercero corrige al segundo, y así sucesivamente.

- ¿Por qué? Al aprender iterativamente de sus propios errores, frecuentemente logra superar en precisión a Random Forest. Es un modelo robusto y altamente competitivo que exprime al máximo la información de los datos estructurados.

Antes de seguir al primer modelo en la Fase 8, hay que hacer un paso previo en el código: separar las variables predictoras ($X$) de la variable variable objetivo ($y$), y transformar las variables de texto (como el Municipio o el Contaminante) en números que los modelos puedan entender matemáticamente (One-Hot Encoding).

### **1. Separación de Variables ($X$ e $y$)**
Todo modelo supervisado necesita separar el mundo en dos:
- La Matriz de Características ($X$): Son todas las pistas que el modelo usará para adivinar. Aquí entran la Variable (PM10 o PM2.5), el Departamento, etc.
- El Vector Objetivo ($y$): Es la respuesta correcta que se quiere predecir. En nuestro caso principal, será la columna Promedio.

### **2. La Transformación: One-Hot Encoding**
Los modelos planteados (Regresión Lineal, Random Forest, Gradient Boosting) no saben qué es "Antioquia" o "PM2.5"; solo procesan matrices numéricas. El One-Hot Encoding toma una columna de texto y la explota en múltiples columnas binarias. Por ejemplo, la columna Variable se convierte en dos columnas nuevas: Variable_PM10 y Variable_PM2.5. Si una fila corresponde a PM10, la columna Variable_PM10 tendrá un 1 y la otra un 0. Esto evita que el modelo crea falsamente que una categoría "vale más" que otra matemáticamente.

### **3. División de Entrenamiento y Prueba (Train-Test Split)**
Si se le enseña al modelo todos los datos, los va a memorizar, y cuando se le muestre un dato nuevo en el futuro, fallará terriblemente (a esto se le llama Overfitting o sobreajuste). Por eso, se usa el dataset limpio y se divide ciegamente:
- Conjunto de Entrenamiento: Los datos con los que los algoritmos de las siguientes fases estudiarán.
- Conjunto de Prueba: Los datos que se ocultaran hasta la Fase 11 para evaluar si el modelo realmente aprendió a generalizar o si solo memorizó.

In [ ]:
from sklearn.model_selection import train_test_split
print("==========================================================")
print("         PREPARACIÓN DE DATOS (ENCODING Y SPLIT)          ")
print("==========================================================\n")

# Definir la variable objetivo (y)
y = df_clean['Promedio']

# Filtrar las variables predictoras (X)
# Eliminar la 'y', las variables que causan Data Leakage y metadata administrativa
columnas_a_eliminar = [
    'Promedio', 'Máximo', 'Mínimo', 'Suma', 'No. de datos', 'Mediana', 'Percentil 98', 
    'Excedencias limite actual', 'Porcentaje excedencias limite actual', 'Días de excedencias',
    'Fechas/horas del máximo', 'Fechas/horas del mínimo', 'Representatividad Temporal',
    'ID Estacion', 'Autoridad Ambiental', 'Estación', 'Tipo de Estación', 'Ubicacion',
    'Código del Departamento', 'Código del Municipio' # Eliminamos los códigos numéricos porque usaremos los Nombres
]

# Se trabajara con 'Variable' (PM10/2.5), 'Año', 'Nombre del Departamento', 'Nombre del Municipio', 'Latitud', 'Longitud'
X_raw = df_clean.drop(columns=columnas_a_eliminar, errors='ignore')

# Transformacion Matemática: One-Hot Encoding
# Convierte todo el texto restante en columnas binarias
X_encoded = pd.get_dummies(X_raw, drop_first=True)

# División de Entrenamiento y Prueba
# test_size=0.2 (20% para examen), random_state=42 (fija la semilla para que los resultados sean reproducibles)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Reporte de transformacion
print("--- REPORTE DE TRANSFORMACIÓN MATEMÁTICA ---")
print(f"Columnas originales en X (con texto): {X_raw.shape[1]}")
print(f"Nuevas columnas binarias creadas: {X_encoded.shape[1]}")
print(f"\n--- DISTRIBUCIÓN DEL DATASET ---")
print(f"Filas para entrenar (X_train): {X_train.shape[0]}")
print(f"Filas ocultas para evaluar (X_test): {X_test.shape[0]}")

### **¿Qué se realizó aquí?**
- **Eliminación de Fuga de Datos (Data Leakage):**
En el EDA (Fase 5) habían variables como el Máximo o las Excedencias están matemáticamente ligadas al Promedio. Si se le proporciona eso al modelo, no estará "prediciendo", estará "haciendo trampa" usando información que en el mundo real no habría al intentar predecir el futuro. Por eso, el código purga todas esas columnas, junto con identificadores que no aportan valor predictivo (como el ID de la estación).

- **La Transformación One-Hot Encoding (pd.get_dummies):**
A un algoritmo no se le puede pasar la palabra "Antioquia" o "PM10". La función get_dummies de Pandas toma una columna de texto y la convierte en columnas binarias. El truco técnico está en usar el parámetro drop_first=True. Esto evita la "Trampa de las Variables Ficticias" (multicolinealidad perfecta). Por ejemplo, si se tiene PM10 y PM2.5, Pandas solo creará una columna llamada Variable_PM2.5. Si vale $1$, el modelo sabe que es PM2.5; si vale $0$, por descarte lógico matemático, sabe que es PM10. Esto le ahorra trabajo al procesador y evita que la Regresión Lineal colapse.

- **El Split de Entrenamiento y Prueba:** 
Con la función train_test_split de la librería scikit-learn para partir la matriz. El 80% de las filas se le darán a los modelos para que aprendan los patrones, y se ocultará el 20% restante para tomarles el examen final en la Fase 11.

## **Fase 8: Entrenamiento de Regresión Lineal (Modelo Base)**

La Regresión Lineal Múltiple intentará encontrar una relación matemática directa trazando un "hiperplano" (una línea plana en 417 dimensiones).La Matemática detrás del Modelo.

El modelo internamente intentará resolver esta ecuación para cada predicción:
$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_{417} x_{417}$ 

Donde:
- $y$ es el Promedio que queremos predecir.
- $\beta_0$ es el punto de partida (intercepto).
- Los $\beta$ son los "pesos" que el modelo le dará a cada una de tus 417 columnas.
- Los $x$ son tus datos (ej. un $1$ si es PM10, un $0$ si es Bogotá, etc.).

In [ ]:
from sklearn.linear_model import LinearRegression

print("==========================================================")
print("    FASE 8.1: ENTRENAMIENTO - REGRESIÓN LINEAL MÚLTIPLE   ")
print("==========================================================\n")

# 1. Inicializar el algoritmo
# Aquí creamos el "cascarón" vacío del modelo matemático
modelo_lr = LinearRegression()

# 2. Entrenar el modelo (Fase de Aprendizaje)
# Le pasamos X_train y y_train para que encuentre la relación lineal entre las variables
print("Entrenando el modelo base con el set de entrenamiento...")
modelo_lr.fit(X_train, y_train)

print("¡Entrenamiento completado exitosamente!")

Una vez que el modelo ya está ajustado y "sabe" cómo se relacionan los datos, hay que ponerlo a prueba. Aquí es donde se usa los datos ocultos (X_test), para que que genere predicciones y calcular matemáticamente qué tanto se equivocó usando el RMSE y el R².

Se usarán dos métricas fundamentales en problemas de regresión:
- RMSE (Root Mean Squared Error - Raíz del Error Cuadrático Medio): Nos dice en promedio por cuántos $\mu g/m^3$ se está equivocando nuestro modelo. Queremos que este número sea lo más bajo posible.
- $R^2$ (Coeficiente de Determinación): Dice qué porcentaje de la variabilidad de la contaminación logra explicar el modelo. Va de 0 a 1. Un $R^2$ de $0.50$ significa que el modelo explica el 50% del fenómeno. Se busca que este número sea lo más alto posible.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

print("==========================================================")
print("           EVALUACION DE METRICAS (REGRESION LINEAL)      ")
print("==========================================================\n")

# 1. Hacer predicciones
# Predecimos sobre el set de PRUEBA (datos nunca vistos por el modelo durante el entrenamiento)
y_pred_lr = modelo_lr.predict(X_test)

# Predecimos también sobre el set de ENTRENAMIENTO (para comparar y verificar si hay sobreajuste)
y_pred_train_lr = modelo_lr.predict(X_train)

# 2. Calcular Métricas de Error (RMSE)
# Extraemos la raíz cuadrada del error cuadrático para que las unidades vuelvan a ser µg/m³
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_lr))
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train_lr))

# 3. Calcular Coeficiente de Determinación (R^2)
# Evaluamos qué porcentaje de la varianza logra explicar el modelo (entre 0 y 1)
r2_test = r2_score(y_test, y_pred_lr)
r2_train = r2_score(y_train, y_pred_train_lr)

# 4. Imprimir los resultados
print("--- RESULTADOS DE MÉTRICAS: REGRESIÓN LINEAL ---")
print(f"RMSE en Entrenamiento: {rmse_train:.2f} µg/m³")
print(f"RMSE en Prueba (Test): {rmse_test:.2f} µg/m³")
print(f"R² en Entrenamiento:   {r2_train:.4f}")
print(f"R² en Prueba (Test):   {r2_test:.4f}")

### **Análisis de la Regresión Lineal (Fase 8):**
- **El lado bueno ($R^2$ de 0.7668):** 
Que un modelo lineal básico logre explicar el 76.6% de la variabilidad de los datos en el conjunto de prueba es un excelente punto de partida. Significa que las variables que seleccionaste (Departamento, Municipio, Contaminante) efectivamente tienen un peso predictivo muy fuerte. Además, al comparar el entrenamiento (0.7841) con la prueba (0.7668), vemos que los valores son casi idénticos. Esto es un triunfo: no hay sobreajuste (overfitting). El modelo generaliza bien. 

- **El lado oscuro (RMSE de 119.12 µg/m³):**
Aquí es donde la Regresión Lineal se rompe. Un error promedio de 119.12 µg/m³ es altísimo para parámetros de calidad del aire (recuerda que el límite permisible de PM2.5 suele rondar los 15 a 50 µg/m³). ¿Por qué ocurre esto? Porque la Regresión Lineal intenta trazar una línea recta, y cuando se topa con esos picos extremos de contaminación que vimos en los boxplots (como los de La Guajira o emergencias ambientales), la línea se desvía bruscamente intentando compensar, generando un margen de error enorme.

## **Fase 9: Entrenamiento de Random Forest Regressor**

Para solucionar el problema del RMSE, se necesita un algoritmo que no trace líneas rectas, sino que divida el espacio mediante lógica condicional. Aquí entra el "Bosque Aleatorio".

¿Qué hace internamente este modelo? En lugar de una sola ecuación matemática, crea cientos de Árboles de Decisión (por defecto, 100). Cada árbol recibe una sub-muestra aleatoria de tus datos y empieza a hacer preguntas de partición. Por ejemplo:
- Nodo 1: ¿Es el Departamento de La Guajira? (Sí/No)
- Nodo 2: Si es Sí, ¿la variable es PM10? (Sí/No)

Al final del árbol (en la "hoja"), se asigna un valor promedio para ese segmento específico. Lo hermoso de este enfoque es que los valores extremos (outliers) quedan aislados en sus propias ramas, sin contaminar ni sesgar las predicciones de los municipios con aire limpio.La predicción final $\hat{y}$ es simplemente el promedio matemático de las predicciones individuales de cada árbol $T_b$:$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$(Donde $B$ es el número total de árboles).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

print("==========================================================")
print("           ENTRENAMIENTO - RANDOM FOREST REGRESSOR        ")
print("==========================================================\n")

# 1. Inicializar el algoritmo
# n_estimators=100 (creará 100 árboles), random_state=42 (para reproducibilidad), n_jobs=-1 (usa todos los núcleos de tu procesador)
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Entrenar el ensamble (Fase de Aprendizaje)
print("Construyendo el bosque aleatorio... (esto puede tardar unos minutos)")
modelo_rf.fit(X_train, y_train)

print("¡Entrenamiento del Random Forest completado exitosamente!")

Ahora entrenado el modelo, hay que evaluarlo con las siguientes métricas.

### **La Trampa del Bosque (Sobreajuste / Overfitting):**
Esta es la métrica de diagnóstico más crítica en un Random Forest. A diferencia de la Regresión Lineal (donde el entrenamiento y la prueba dieron casi igual: 0.78 vs 0.76), el Random Forest tiene una tendencia natural y agresiva a "memorizar" los datos si no se le ponen límites.

- Alerta Roja: Si hay un R² en el set de Entrenamiento cercano a 0.98 o 0.99, pero el R² en Prueba se queda estancado en los 0.70s. Eso significa que el modelo se aprendió las respuestas de memoria, pero fallará en el mundo real.

- Lo Esperado: Una brecha moderada (por ejemplo, 0.92 en entrenamiento y 0.84 en prueba) es completamente normal y aceptable en la arquitectura de este ensamble.

### **El Combate contra el RMSE (El error absoluto):**
El "enemigo a vencer" es el RMSE de 119.12 µg/m³ que dejó la Regresión Lineal en el set de prueba.

Como el Random Forest aísla los valores atípicos (los picos extremos de contaminación por incendios o minería) en ramas específicas en lugar de intentar trazar una línea recta rígida, debería haber una caída drástica en este número. Si baja, confirmamos empíricamente que la arquitectura no lineal era la respuesta al problema de los outliers.

### **El R² (El techo de cristal):**
El piso a superar es el 0.7668 (76.68%). Al capturar relaciones complejas e interacciones entre variables (por ejemplo, el modelo aprende reglas como: "Si es Bogotá Y es PM2.5 Y es 2024, entonces el comportamiento es X"), el porcentaje de explicación de la realidad (R²) debería dispararse hacia arriba.

In [ ]:
print("==========================================================")
print("         EVALUACION DE METRICAS (RANDOM FOREST)           ")
print("==========================================================\n")

# 1. Hacer predicciones
# Prediccion en el set de PRUEBA
y_pred_rf = modelo_rf.predict(X_test)

# Prediccion en el set de ENTRENAMIENTO
y_pred_train_rf = modelo_rf.predict(X_train)

# 2. Calcular Metricas de Error (RMSE)
rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_train_rf))

# 3. Calcular Coeficiente de Determinacion (R^2)
r2_test_rf = r2_score(y_test, y_pred_rf)
r2_train_rf = r2_score(y_train, y_pred_train_rf)

# 4. Imprimir los resultados
print("--- RESULTADOS DE MÉTRICAS: RANDOM FOREST ---")
print(f"RMSE en Entrenamiento: {rmse_train_rf:.2f} µg/m³")
print(f"RMSE en Prueba (Test): {rmse_test_rf:.2f} µg/m³")
print(f"R² en Entrenamiento:   {r2_train_rf:.4f}")
print(f"R² en Prueba (Test):   {r2_test_rf:.4f}")

### **Análisis de Random Forest Regressor (Fase 9):**
- **La Victoria Absoluta (El colapso del RMSE y el salto del R²):**
    - El salto en la explicación: Se pasó de un $R^2$ de 0.76 en prueba con la regresión lineal, a un sorprendente 0.9598. Esto significa que el Random Forest es capaz de explicar prácticamente el 96% del comportamiento de la contaminación en Colombia. Es un modelo con un poder predictivo altísimo.
    - La derrota del error: El RMSE cayó drásticamente de 119.12 µg/m³ a 49.48 µg/m³ en los datos no vistos. La hipótesis metodológica fue correcta: al usar árboles que segmentan los datos con preguntas de "sí/no", el modelo logró aislar esos outliers geográficos (como la minería en La Guajira o el tráfico en Bogotá) en ramas específicas, impidiendo que distorsionaran la predicción de los municipios más limpios.

- **La Trampa Confirmada (El diagnóstico de Sobreajuste):**
Al observar detenidamente la brecha entre el entrenamiento y la prueba, se ve el comportamiento más clásico (y peligroso) del Random Forest:
    - Entrenamiento: $R^2$ de 0.9956 y RMSE de 15.80.
    - Prueba: $R^2$ de 0.9598 y RMSE de 49.48.
    
    El modelo prácticamente memorizó el set de entrenamiento (un 99.5% de precisión es casi imposible en la naturaleza sin hacer trampa). Como se le puso límites al algoritmo, los árboles crecieron de forma descontrolada hasta crear ramas específicas para casi cada registro individual.
    ¿Es un modelo inútil entonces? A pesar de haberse aprendido el entrenamiento de memoria, logró generalizar excepcionalmente bien (ese 96% en prueba es innegable). Sin embargo, este grado de sobreajuste (overfitting) nos indica que el modelo es ineficiente.

## **Fase 10: Entrenamiento de Gradient Boosting Regressor**

Mientras que el Random Forest crea 100 árboles de forma independiente y paralela, el Gradient Boosting trabaja de forma estrictamente secuencial:

- El Árbol 1 hace una primera predicción, usualmente bastante mediocre.
- El algoritmo calcula los errores (residuales) de ese primer árbol. Es decir, dónde se equivocó y por cuánto.
- El Árbol 2 no intenta predecir la contaminación desde cero; su único trabajo es intentar predecir y corregir los errores que cometió el Árbol 1.
- El Árbol 3 corrige los errores del Árbol 2, y así sucesivamente.

En lugar de crear árboles profundos que memorizan los datos (como hizo nuestro Random Forest), el Gradient Boosting usa árboles muy pequeños y débiles (a menudo de solo 3 niveles de profundidad). Al aprender lentamente y paso a paso, suele generalizar mucho mejor ante datos nunca vistos. 

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

print("==========================================================")
print("             ENTRENAMIENTO - GRADIENT BOOSTING          ")
print("==========================================================\n")

# 1. Inicializar el algoritmo
# n_estimators=100 (100 pasos de correccion), learning_rate=0.1 (qué tan rapido aprende de sus errores)
modelo_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

# 2. Entrenar el ensamble secuencial (Fase de Aprendizaje)
print("Entrenando el modelo Gradient Boosting (iterando errores)...")
modelo_gb.fit(X_train, y_train)

print("¡Entrenamiento del Gradient Boosting completado exitosamente!")

Terminado el entrenamiento de Gradient Boosting, se establecen las métricas para evaularlo:

### **1. La Prueba de Fuego (El Control del Sobreajuste):**
Teniendo en cuenta que el Random Forest sacó un $R^2$ de 0.99 en entrenamiento y 0.95 en prueba; es decir, memorizó los datos a la fuerza. Lo que se busca ahora: Como el Gradient Boosting usa árboles pequeños y controlados, se desea ver una brecha mucho más estrecha. Por ejemplo, un $0.97$ en entrenamiento y un $0.96$ en prueba, será una victoria absoluta. Significa que el modelo aprendió los patrones reales sin tener que memorizar el dataset.

### **2. El Nuevo Piso a Romper (RMSE):**
El Random Forest dejó un error promedio bastante decente de 49.48 µg/m³ (derrotando los 119 de la regresión lineal).Ahora: Se quiere ver si este enfoque de "corregir el error del árbol anterior" logra exprimir aún más la precisión y bajar ese RMSE a los 40s o incluso menos. Cada microgramo que baje significa que el modelo es mucho más confiable para alertar sobre emergencias ambientales.

In [ ]:
print("==========================================================")
print("         EVALUACION DE METRICAS (GRADIENT BOOSTING)      ")
print("==========================================================\n")

# 1. Hacer predicciones
# Prediccion en el set de PRUEBA
y_pred_gb = modelo_gb.predict(X_test)

# Prediccion en el set de ENTRENAMIENTO
y_pred_train_gb = modelo_gb.predict(X_train)

# 2. Calcular Metricas de Error (RMSE)
rmse_test_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
rmse_train_gb = np.sqrt(mean_squared_error(y_train, y_pred_train_gb))

# 3. Calcular Coeficiente de Determinacion (R^2)
r2_test_gb = r2_score(y_test, y_pred_gb)
r2_train_gb = r2_score(y_train, y_pred_train_gb)

# 4. Imprimir los resultados
print("--- RESULTADOS DE METRICAS: GRADIENT BOOSTING ---")
print(f"RMSE en Entrenamiento: {rmse_train_gb:.2f} µg/m³")
print(f"RMSE en Prueba (Test): {rmse_test_gb:.2f} µg/m³")
print(f"R² en Entrenamiento:   {r2_train_gb:.4f}")
print(f"R² en Prueba (Test):   {r2_test_gb:.4f}")

### **Análisis de Gradient Boosting Regressor (Fase 10):**
Los resultados anteriores represetan un caso severo de Subajuste (Underfitting):
- El colapso del $R^2$: Caer a un ~57% en la prueba significa que el modelo perdió casi la mitad de su capacidad predictiva frente al Random Forest.
- El retroceso del RMSE: Un error de 161.91 µg/m³ es inaceptable. El modelo está disparando a ciegas frente a los picos de contaminación.
- ¿Por qué falló tan estrepitosamente? El Gradient Boosting viene configurado por defecto para crear árboles muy superficiales (usualmente de 3 niveles de profundidad). Al pasarle 417 columnas (la inmensa mayoría llenas de ceros y unos por el One-Hot Encoding), esos árboles tan pequeños se quedaron sin capacidad para encontrar los patrones. Tuvo que resolver un laberinto gigante dando pasos minúsculos (solo 100 árboles). No memorizó los datos, de hecho, no aprendió casi nada.

## **Fase 11: Optimización General (Plan de Contingencia)**
Los 3 modelos tuvieron problemas, usar uno sería ignorar las fallas que tuvo y caer en predicciones erroneas. Incluso usar el mejor, representaría un fallo predictivo y estructural bastante grande. Por esto, el plan a seguir es reparar y mejorar estos modelos para que funcionen de mejor manera. La idea es aplicarlo con todos, y posteriormente obtener el verdadero ganador. Para esto, hay que seguir dos caminos importantes:
- Camino 1: Se evidenció empíricamente que el Random Forest tiene un potencial sorprendente (llegó casi al 96% de explicación y bajó el error a 49 µg/m³), su única falencia fue el sobreajuste. Se puede abrir una fase de optimización y usar GridSearchCV para "podar" esos árboles, limitando su profundidad máxima. Si se logra que deje de memorizar el entrenamiento, ese RMSE en prueba va a caer aún más.
- Camino 2: El problema de fondo que está rompiendo a la Regresión Lineal y al Gradient Boosting son los outliers extremos de contaminación. Si se le aplica un logaritmo a la variable $y$ (Promedio) antes de entrenar, se suaviza esa curva extrema. Los modelos entrenarán sobre una campana mucho más normalizada, y luego simplemente habrá que revertir el logaritmo a las predicciones finales. Es un "truco" matemático que suele hacer magia.

### **1. Optimización de la Regresión Lineal**
#### **Transformación Logarítmica:**
El modelo de Regresión Lineal base presentó un error elevado debido a su sensibilidad ante valores atípicos (outliers). Los datos de calidad del aire poseen una distribución asimétrica positiva, con la mayoría de los registros agrupados en valores bajos y algunos picos extremos de contaminación.Para corregir esto, se aplicará una transformación logarítmica, específicamente $\log(y+1)$, a la variable objetivo (Promedio) antes del entrenamiento. Esta operación matemática comprime la escala de los valores extremos, normalizando la distribución de los datos y permitiendo que el algoritmo trace un hiperplano más ajustado. Una vez el modelo genera las predicciones en escala logarítmica, se aplica la función inversa (exponencial) para retornar los valores a su unidad original ($\mu g/m^3$) y calcular las métricas reales.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("==========================================================")
print("         OPTIMIZACION - REGRESION LINEAL (LOG SCALE)      ")
print("==========================================================\n")

# 1. Transformacion Logaritmica de la variable objetivo (y)
# Se usa log1p que matematicamente es log(1 + x) para evitar errores con valores en 0
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 2. Entrenamiento del modelo optimizado
modelo_lr_opt = LinearRegression()
modelo_lr_opt.fit(X_train, y_train_log)

# 3. Prediccion (los resultados estarán en escala logarítmica)
y_pred_log_test = modelo_lr_opt.predict(X_test)
y_pred_log_train = modelo_lr_opt.predict(X_train)

# 4. Reversion de la transformación (Exponencial)
# Se usa expm1 que matemáticamente es exp(x) - 1 para volver a las unidades de µg/m³
y_pred_test_opt = np.expm1(y_pred_log_test)
y_pred_train_opt = np.expm1(y_pred_log_train)

# 5. Calculo de Metricas Finales
rmse_test_lr_opt = np.sqrt(mean_squared_error(y_test, y_pred_test_opt))
rmse_train_lr_opt = np.sqrt(mean_squared_error(y_train, y_pred_train_opt))

r2_test_lr_opt = r2_score(y_test, y_pred_test_opt)
r2_train_lr_opt = r2_score(y_train, y_pred_train_opt)

# 6. Impresion de resultados
print("--- RESULTADOS METRICAS: REGRESION LINEAL OPTIMIZADA ---")
print(f"RMSE en Entrenamiento: {rmse_train_lr_opt:.2f} µg/m³")
print(f"RMSE en Prueba (Test): {rmse_test_lr_opt:.2f} µg/m³")
print(f"R² en Entrenamiento:   {r2_train_lr_opt:.4f}")
print(f"R² en Prueba (Test):   {r2_test_lr_opt:.4f}")

#### **Análisis de las Métricas (Regresión Lineal Optimizada):**
Al analizar los resultados arrojados por la consola, se observa que la transformación logarítmica no generó el impacto positivo esperado sobre este algoritmo en particular. Por el contrario, se evidencia una leve degradación en su capacidad predictiva en comparación con el modelo base de la Fase 8.
- Comportamiento del R²: El porcentaje de varianza explicada en el conjunto de prueba disminuyó ligeramente, pasando de 0.7668 a 0.7544.
- Comportamiento del RMSE: El error promedio aumentó de 119.12 µg/m³ a 122.24 µg/m³.
Justificación técnica: Este fenómeno ocurre porque la Regresión Lineal asume relaciones estrictamente proporcionales. Aunque el logaritmo normaliza la variable objetivo reduciendo la influencia de los valores extremos durante el entrenamiento, cualquier pequeño margen de error que el modelo cometa en la escala logarítmica se magnifica exponencialmente al momento de aplicar la función inversa (np.expm1) para retornar a las unidades originales de microgramos por metro cúbico. Se concluye que la rigidez geométrica del modelo lineal no se beneficia de esta transformación para este conjunto de datos específico

### **2. Optimización de Random Forest Regressor**
En la Fase 9, el Random Forest demostró un excelente rendimiento en el conjunto de prueba (RMSE de 49.48 µg/m³), pero evidenció un severo caso de sobreajuste (memorización), alcanzando un R² de 0.9956 en el entrenamiento.

Para optimizar este modelo y garantizar que aprenda patrones generalizables en lugar de memorizar el ruido, se procederá a implementar técnicas de "poda" mediante el ajuste de hiperparámetros. Específicamente, se restringirá la arquitectura del bosque utilizando dos parámetros clave:

- **max_depth:** Limita la profundidad máxima de cada árbol. Al evitar que crezcan infinitamente, se impide que creen ramas exclusivas para datos individuales.
- **min_samples_leaf:** Establece el número mínimo de registros que debe tener una hoja final para ser considerada válida. Esto fuerza al algoritmo a agrupar los datos, promediando el comportamiento en lugar de aislar casos atípicos únicos.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("==========================================================")
print("     OPTIMIZACION - RANDOM FOREST (PODA DE ARBOLES)       ")
print("==========================================================\n")

# 1. Configuracion del modelo con hiperparametros restrictivos
# Se limita la profundidad a 20 niveles y se exigen al menos 5 datos por hoja
modelo_rf_opt = RandomForestRegressor(
    n_estimators=100, 
    max_depth=20, 
    min_samples_leaf=5, 
    random_state=42, 
    n_jobs=-1
)

# 2. Entrenamiento del modelo optimizado (usando los datos originales)
print("Entrenando Random Forest con restricciones de arquitectura...")
modelo_rf_opt.fit(X_train, y_train)

# 3. Predicciones
y_pred_rf_opt_test = modelo_rf_opt.predict(X_test)
y_pred_rf_opt_train = modelo_rf_opt.predict(X_train)

# 4. Calculo de Metricas Finales
rmse_test_rf_opt = np.sqrt(mean_squared_error(y_test, y_pred_rf_opt_test))
rmse_train_rf_opt = np.sqrt(mean_squared_error(y_train, y_pred_rf_opt_train))

r2_test_rf_opt = r2_score(y_test, y_pred_rf_opt_test)
r2_train_rf_opt = r2_score(y_train, y_pred_rf_opt_train)

# 5. Impresion de resultados
print("--- RESULTADOS METRICAS: RANDOM FOREST OPTIMIZADO ---")
print(f"RMSE en Entrenamiento: {rmse_train_rf_opt:.2f} µg/m³")
print(f"RMSE en Prueba (Test): {rmse_test_rf_opt:.2f} µg/m³")
print(f"R² en Entrenamiento:   {r2_train_rf_opt:.4f}")
print(f"R² en Prueba (Test):   {r2_test_rf_opt:.4f}")

### **Analisis de las Métricas (Random Forest Regressor):**
Los resultados evidencian que la restricción de hiperparámetros cumplió su objetivo metodológico al mitigar el sobreajuste (overfitting).
- Control del sobreajuste: La brecha en la métrica $R^2$ entre el conjunto de entrenamiento (0.9431) y el de prueba (0.9223) se redujo a un margen cercano al 2%. Esto confirma que el algoritmo dejó de memorizar los datos y ahora es capaz de generalizar los patrones de contaminación atmosférica.
- El balance técnico (Trade-off): Al limitar la profundidad de los árboles (max_depth) y exigir un mínimo de muestras por hoja (min_samples_leaf), se le restó flexibilidad al modelo. Esto resultó en un incremento del RMSE en el conjunto de prueba, pasando de 49.48 µg/m³ a 68.76 µg/m³.

En términos de ingeniería de datos, este es un modelo superior al de la Fase 9, ya que es estadísticamente más robusto y confiable para un entorno de producción, aunque el costo haya sido sacrificar un margen de precisión absoluta.

### **3. Optimización de Gradient Boosting**
#### **Transformación Logarítmica + Expansión de Capacidad:**
En la Fase 10, el Gradient Boosting base presentó un caso severo de subajuste (underfitting). El modelo fue incapaz de procesar la alta dimensionalidad (más de 400 variables tras el One-Hot Encoding) utilizando sus parámetros por defecto, los cuales construyen árboles muy poco profundos. Para esta optimización, se implementará una estrategia dual:
- Suavizado Matemático: Se aplicará nuevamente la transformación logarítmica $\log(y+1)$ a la variable objetivo. A diferencia de la Regresión Lineal, los algoritmos basados en árboles secuenciales son muy sensibles a la función de pérdida; reducir el impacto de los valores atípicos (outliers) facilita la convergencia del modelo.
- Expansión Arquitectónica: Se incrementará la capacidad del algoritmo ajustando max_depth a 5 (para que pueda hacer más cruces de variables geográficas y temporales) y n_estimators a 200 (para darle más iteraciones en la corrección de errores).

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

print("==========================================================")
print("     OPTIMIZACION - GRADIENT BOOSTING (LOG + TUNE)        ")
print("==========================================================\n")

# 1. Transformacion Logaritmica de la variable objetivo (y)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 2. Configuracion del modelo con mayor capacidad de aprendizaje
modelo_gb_opt = GradientBoostingRegressor(
    n_estimators=200, 
    max_depth=5, 
    learning_rate=0.1,
    random_state=42
)

# 3. Entrenamiento del modelo optimizado (usando los datos transformados)
print("Entrenando Gradient Boosting expandido en escala logarítmica...")
modelo_gb_opt.fit(X_train, y_train_log)

# 4. Predicciones (los resultados estarán en escala logaritmica)
y_pred_log_gb_test = modelo_gb_opt.predict(X_test)
y_pred_log_gb_train = modelo_gb_opt.predict(X_train)

# 5. Reversion de la transformación (Exponencial) para volver a µg/m³
y_pred_gb_opt_test = np.expm1(y_pred_log_gb_test)
y_pred_gb_opt_train = np.expm1(y_pred_log_gb_train)

# 6. Calculo de Metricas Finales
rmse_test_gb_opt = np.sqrt(mean_squared_error(y_test, y_pred_gb_opt_test))
rmse_train_gb_opt = np.sqrt(mean_squared_error(y_train, y_pred_gb_opt_train))

r2_test_gb_opt = r2_score(y_test, y_pred_gb_opt_test)
r2_train_gb_opt = r2_score(y_train, y_pred_gb_opt_train)

# 7. Impresion de resultados
print("--- RESULTADOS METRICAS: GRADIENT BOOSTING OPTIMIZADO ---")
print(f"RMSE en Entrenamiento: {rmse_train_gb_opt:.2f} µg/m³")
print(f"RMSE en Prueba (Test): {rmse_test_gb_opt:.2f} µg/m³")
print(f"R² en Entrenamiento:   {r2_train_gb_opt:.4f}")
print(f"R² en Prueba (Test):   {r2_test_gb_opt:.4f}")

#### **Análisis de las Métricas (Gradient Boosting Regressor):**
Los resultados indican que, a pesar de la expansión de la capacidad arquitectónica (max_depth=5, n_estimators=200) y la normalización de la variable objetivo mediante transformación logarítmica, el modelo de Gradient Boosting continúa presentando un desempeño inferior en comparación con las variantes de Random Forest evaluadas.
- Capacidad Predictiva: Se obtuvo un $R^2$ de 0.6642 en el conjunto de prueba. Si bien esto representa una mejora respecto al modelo de Gradient Boosting base (Fase 10), es insuficiente para equiparar la precisión alcanzada por el Random Forest.
- Error de Predicción: El RMSE de prueba se sitúa en 142.93 µg/m³, un margen de error considerablemente mayor a los 68.76 µg/m³ obtenidos por el Random Forest optimizado.

Interpretación técnica: La arquitectura secuencial del Gradient Boosting, incluso tras las optimizaciones, muestra una convergencia más lenta y menos efectiva para este conjunto de datos. Probablemente, la alta dimensionalidad derivada del One-Hot Encoding genera una dispersión de datos que el enfoque de aprendizaje residual del Gradient Boosting no logra capturar con la misma eficacia que el enfoque de promediado de árboles independientes del Random Forest.

### **Conclusión de la Fase de Optimización: Selección del Modelo Definitivo**
            Modelo:                  RMSE(Test):             R^2(Test):          Estado del Modelo:
        Regresion Lineal            122.24 µg/m^3             0.7544                Subajustado
          Random Forest              68.76 µg/m^3             0.9223             Generalizado(Óptimo)
        Gradient Boosting           142.93 µg/m^3             0.6642                Subajustado


El Random Forest (Optimizado) se establece como el modelo definitivo del proyecto.
- Fiabilidad estadística: Es el único modelo que supera el umbral del 90% en $R^2$, manteniendo una brecha mínima con el conjunto de entrenamiento (ausencia de sobreajuste).
- Precisión operativa: Su RMSE de 68.76 µg/m³ es significativamente más robusto, permitiendo una capacidad de predicción superior ante variaciones geográficas y temporales en la calidad del aire.
- Estabilidad: A diferencia de los otros modelos, demostró una consistencia sólida incluso tras aplicar restricciones para prevenir la memorización de datos.

## **Fase 12: Persistencia del Modelo y Validación de Inferencia**
El objetivo es transformar el objeto de Python en un archivo binario (.pkl) que pueda ser transportado y cargado en cualquier servidor. Además, se realizará una prueba de integridad: cargar el archivo guardado y solicitarle una predicción para verificar que el modelo no perdió sus "conocimientos" ni su precisión durante la serialización.

In [ ]:
import joblib

print("==========================================================")
print("         PERSISTENCIA Y VALIDACION DEL MODELO            ")
print("==========================================================\n")

# 1. Guardar el modelo ganador (Random Forest Optimizado)
nombre_archivo = 'modelo_calidad_aire_rf.pkl'
joblib.dump(modelo_rf_opt, nombre_archivo)
print(f"Modelo serializado y guardado exitosamente como: {nombre_archivo}")

# 2. Simular la carga del modelo
print("\nCargando el modelo desde el archivo .pkl...")
modelo_cargado = joblib.load(nombre_archivo)

# 3. Prueba de Inferencia: Verificar que el modelo cargado predice igual que el original
# Se toma un registro de prueba como ejemplo
muestra_prueba = X_test[0:1] 
prediccion_original = modelo_rf_opt.predict(muestra_prueba)
prediccion_cargada = modelo_cargado.predict(muestra_prueba)

print(f"Prediccion original: {prediccion_original[0]:.4f} µg/m³")
print(f"Prediccion cargada:  {prediccion_cargada[0]:.4f} µg/m³")

if np.isclose(prediccion_original, prediccion_cargada):
    print("\n[EXITO] La validacion fue exitosa: El modelo cargado es identico al original.")
else:
    print("\n[ERROR] La validación fallo: Existe discrepancia en las predicciones.")

## **Resumen Ejecutivo: Proyecto de Modelado Predictivo de Calidad del Aire**
A continuación, se presenta la síntesis de la metodología aplicada, diseñada para validar la viabilidad técnica ante los responsables del proyecto.

### **1. Objetivo y Alcance**
El proyecto tuvo como fin desarrollar un modelo predictivo capaz de estimar concentraciones de material particulado ($\mu g/m^3$) en diversos departamentos y municipios de Colombia, superando las limitaciones de los modelos lineales básicos mediante el uso de algoritmos de ensamble.

### **2. Metodología de Ingeniería**
El proceso se estructuró en tres ejes fundamentales:
- Exploración y Preprocesamiento: Se realizó una limpieza exhaustiva y transformación de datos, incluyendo One-Hot Encoding para variables categóricas, abordando una alta dimensionalidad superior a las 400 columnas.
- Arquitectura de Modelos: Se compararon tres enfoques distintos:
    - Regresión Lineal: Evaluada como línea base, resultó insuficiente debido a su rigidez frente a la distribución asimétrica de los datos.
    - Gradient Boosting: A pesar de su potencia teórica, la alta dimensionalidad y la escala de los datos limitaron su capacidad de aprendizaje inicial, incluso tras optimizaciones.
    - Random Forest: Identificado como el modelo de mejor desempeño. Se aplicaron técnicas de poda (max_depth, min_samples_leaf) para erradicar el sobreajuste (overfitting), logrando un modelo robusto y generalizable.
- Validación y Persistencia: Se implementó una metodología de prueba estricta para garantizar que el modelo no dependiera de la memorización de registros, culminando con la serialización mediante joblib para su integración en servicios de API.

### **3. Conclusiones y Próximos Pasos**
El modelo seleccionado representa un equilibrio óptimo entre precisión técnica y estabilidad operativa. La validación mediante persistencia binaria (.pkl) asegura que el "corazón" de la solución es trasladable y reproducible.

Como siguiente paso, se recomienda la integración del archivo modelo_calidad_aire_rf.pkl en el entorno de desarrollo del backend, utilizando el mismo pipeline de preprocesamiento (escalado y codificación) aplicado durante la etapa de entrenamiento para garantizar la coherencia en las futuras predicciones en tiempo real.